# Day 23 — Baseline: Logistic Regression on windows (patient-ID split)

Status: COMPLETE — `src/baseline.py` trained on 620,401 rows (16,268 patients),
tested on 154,468 rows (4,068 patients). Model + metrics saved; 7/7 tests pass.

**The one rule of this day:** split by **patient ID**, never by row. Random row
shuffling puts a patient's future hours in train and past hours in test — the
model would 'recognize' patients instead of learning physiology.

In [1]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
from baseline import exclude_post_onset, split_by_patient  # noqa: E402

df = pd.read_parquet("../data/windows_setA.parquet")
df = exclude_post_onset(df)
train, test = split_by_patient(df)
assert set(train["pid"]).isdisjoint(set(test["pid"]))
print(f"train: {len(train)} rows / {train['pid'].nunique()} patients")
print(f"test:  {len(test)} rows / {test['pid'].nunique()} patients")
print(f"test positive rate: {test['SepsisLabel'].mean()*100:.3f}% "
      "(only the onset hour per septic patient is positive)")
print("post-onset hours excluded; no patient appears on both sides")

train: 620401 rows / 16268 patients
test:  154468 rows / 4068 patients
test positive rate: 0.232% (only the onset hour per septic patient is positive)
post-onset hours excluded; no patient appears on both sides


In [2]:
# Training lives in src/baseline.py (run: .venv/bin/python src/baseline.py).
# Here we load and interpret its saved metrics.
import json
from pathlib import Path

m = json.loads(Path("../models/baseline_metrics.json").read_text())
print(f"ROC-AUC:  {m['roc_auc']}  (ranks septic hours above clean ones, modestly)")
print(f"PR-AUC:   {m['pr_auc']}  (~2.6x random at 0.23% prevalence — the honest floor)")
print(f"F1@0.5:   {m['f1_at_0.5']}  | precision {m['precision_at_0.5']}, "
      f"recall {m['recall_at_0.5']}")
c = m["confusion_at_0.5"]
print(f"confusion: tn={c['tn']} fp={c['fp']} fn={c['fn']} tp={c['tp']}")
print("model: models/lr_baseline.joblib "
      "(median-impute + scale + balanced LR, 96 feats)")

ROC-AUC:  0.7237  (ranks septic hours above clean ones, modestly)
PR-AUC:   0.0060  (~2.6x random at 0.23% prevalence — the honest floor)
F1@0.5:   0.0093  | precision 0.0047, recall 0.662
confusion: tn=103940 fp=50170 fn=121 tp=237
model: models/lr_baseline.joblib (median-impute + scale + balanced LR, 96 feats)


## Findings (the floor is set — now beat it)

1. **Ranking works a little (0.72), operating point is unusable.** At 0.5 the
   model flags 50k clean hours to catch 237 septic ones (precision 0.5%) —
   textbook alert fatigue. Threshold tuning (Day 25) and better models (Day 24)
   close this gap; the baseline's job is to *measure* it, not solve it.
2. **Positives are scarce by design:** 0.23% because only the onset hour counts.
   PR-AUC 0.006 vs 0.0023 random is the number to beat, not ROC-AUC.
3. **Median-imputation was forced by LR** (linear models need dense input) — it
   dilutes the missingness signal from Day 22. Day 24's trees use NaNs natively.

## Handoff to Day 24

Same windows, same split: LightGBM on engineered features vs a sequence model
(LSTM/1D-CNN). Expect trees to win on this data scale — sequences need more
data to learn temporal patterns the rolling features already encode.